# Notebook 4: Evaluating Generated Materials — MLIPs

**APS Tutorial T4 — Generative AI for Physics: From Models to Materials**

In this notebook you will learn to:
- Understand why **evaluation** is critical after generation
- Use a **Machine Learning Interatomic Potential** (CHGNet) to evaluate structures
- Predict energies, forces, and stresses in seconds (vs. hours with DFT)
- Relax a crystal structure using an MLIP
- Connect evaluation to SCIGEN's screening pipeline

> **Prerequisites:** Run `00_setup.ipynb` first (CHGNet was installed there).

In [ ]:
# --- Run once per Colab session (skip if you already ran 00_setup.ipynb) ---
import importlib, subprocess, sys

def _ensure(pkg, pip_name=None):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pip_name or pkg])

_ensure('pymatgen')
_ensure('chgnet')

# Clone the repo if not present (needed for dataset files)
import os, shutil
REPO_URL = 'https://github.com/RyotaroOKabe/APS_demo_SCIGEN.git'
PROJECT_DIR = '/content/APS_demo_SCIGEN'
if not os.path.exists(os.path.join(PROJECT_DIR, '.git')):
    if os.path.exists(PROJECT_DIR):
        shutil.rmtree(PROJECT_DIR)
    os.system(f'git clone {REPO_URL} {PROJECT_DIR}')
os.environ.setdefault('PROJECT_ROOT', PROJECT_DIR)
print('Ready.')

---
## 4.1 The evaluation problem

Generating crystal structures is only half the job. After generation, we need to ask:

1. **Is this structure physically reasonable?** (sensible bond lengths, coordination)
2. **Is it thermodynamically stable?** (low energy, near the convex hull)
3. **Does it have interesting properties?** (magnetic, electronic, mechanical)

### The traditional approach: DFT
Density Functional Theory (DFT) gives accurate energies and forces, but it's
**expensive** — each structure takes minutes to hours on a supercomputer.
If you generated 10,000 candidates, DFT relaxation could take weeks.

### The modern approach: MLIPs
Machine Learning Interatomic Potentials are trained on DFT data and can predict
energies/forces/stresses in **seconds** — 1000x faster than DFT.
They enable rapid screening of large candidate sets.

---
## 4.2 CHGNet: a universal MLIP

[CHGNet](https://github.com/CederGroupHub/chgnet) (Crystal Hamiltonian Graph Neural Network)
is a universal potential trained on the Materials Project database.
It can predict energies, forces, stresses, and magnetic moments for any inorganic material.

Let's load the pretrained model.

In [ ]:
from chgnet.model.model import CHGNet

chgnet = CHGNet.load()
print(f'CHGNet loaded: {sum(p.numel() for p in chgnet.parameters()):,} parameters')

---
## 4.3 Predicting properties

Let's predict the energy of a known structure from the training data.

In [ ]:
import os
import pandas as pd
from pymatgen.core import Structure

PROJECT_DIR = os.environ.get('PROJECT_ROOT', '/content/APS_demo_SCIGEN')
df = pd.read_csv(os.path.join(PROJECT_DIR, 'data', 'mp_20', 'test.csv'))

# Pick a structure
row = df.iloc[0]
struct = Structure.from_str(row['cif'], fmt='cif')

print(f'Structure: {row["pretty_formula"]} ({row["material_id"]})')
print(f'Atoms: {struct.num_sites}')
print(f'DFT formation energy: {row["formation_energy_per_atom"]:.4f} eV/atom')

In [ ]:
# Predict with CHGNet
prediction = chgnet.predict_structure(struct)

print(f'CHGNet predictions for {row["pretty_formula"]}:')
print(f'  Energy:  {prediction["e"]:>10.4f} eV/atom')
print(f'  Forces:  max |F| = {abs(prediction["f"]).max():.4f} eV/Å')
print(f'  Stress:  {prediction["s"].diagonal().tolist()}')
if 'm' in prediction:
    print(f'  Magmom:  {prediction["m"].tolist()}')

---
## 4.4 Structure relaxation

Relaxation optimizes the atomic positions and lattice to minimize the energy.
This is like letting the structure "settle" into its lowest-energy configuration.

With CHGNet, this takes seconds instead of hours.

In [ ]:
from chgnet.model.dynamics import StructOptimizer

optimizer = StructOptimizer()

# Relax the structure
result = optimizer.relax(struct, verbose=True)

relaxed = result['final_structure']
print(f'\nRelaxation complete.')
print(f'  Initial energy: {prediction["e"]:.4f} eV/atom')
print(f'  Final energy:   {result["trajectory"].energies[-1] / relaxed.num_sites:.4f} eV/atom')

In [ ]:
# Compare original and relaxed structures
import numpy as np

print('Lattice parameter changes:')
for param in ['a', 'b', 'c', 'alpha', 'beta', 'gamma']:
    orig = getattr(struct.lattice, param)
    relax = getattr(relaxed.lattice, param)
    unit = 'Å' if param in ['a', 'b', 'c'] else '°'
    print(f'  {param:>5s}: {orig:8.3f} → {relax:8.3f} {unit}  (Δ = {relax - orig:+.3f})')

# Atomic displacement
displacements = np.linalg.norm(
    relaxed.cart_coords - struct.cart_coords, axis=1
)
print(f'\nMax atomic displacement: {displacements.max():.4f} Å')
print(f'Mean atomic displacement: {displacements.mean():.4f} Å')

---
## 4.5 Batch evaluation

One of the key advantages of MLIPs is speed. Let's evaluate many structures quickly.

In [ ]:
import time

# Evaluate the first 20 structures in the test set
n_eval = min(20, len(df))
energies_chgnet = []
energies_dft = []
formulas = []

start = time.time()
for i in range(n_eval):
    row = df.iloc[i]
    try:
        s = Structure.from_str(row['cif'], fmt='cif')
        pred = chgnet.predict_structure(s)
        energies_chgnet.append(float(pred['e']))
        energies_dft.append(row['formation_energy_per_atom'])
        formulas.append(row['pretty_formula'])
    except Exception as e:
        print(f'  Skipping {row["pretty_formula"]}: {e}')

elapsed = time.time() - start
print(f'Evaluated {len(energies_chgnet)} structures in {elapsed:.1f}s')
print(f'({elapsed/len(energies_chgnet)*1000:.0f} ms per structure)')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(6, 6))

ax.scatter(energies_dft, energies_chgnet, s=50, c='steelblue',
           edgecolors='white', linewidths=0.5)

# Diagonal line
lims = [min(min(energies_dft), min(energies_chgnet)) - 0.2,
        max(max(energies_dft), max(energies_chgnet)) + 0.2]
ax.plot(lims, lims, 'k--', alpha=0.3, label='Perfect agreement')

# Compute MAE
mae = np.mean(np.abs(np.array(energies_dft) - np.array(energies_chgnet)))

ax.set_xlabel('DFT formation energy (eV/atom)', fontsize=12)
ax.set_ylabel('CHGNet energy (eV/atom)', fontsize=12)
ax.set_title(f'CHGNet vs. DFT\nMAE = {mae:.3f} eV/atom', fontsize=13)
ax.legend()
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

print(f'Note: CHGNet predicts total energy, not formation energy.')
print(f'The offset is expected — what matters is that the TREND is captured.')

---
## 4.6 Connection to SCIGEN's screening pipeline

SCIGEN includes its own GNN-based screening models (in `gnn_eval/`) that evaluate:
- **Stability** — Is the structure near the convex hull?
- **Magnetism** — Does the structure have interesting magnetic properties?

The full SCIGEN pipeline:
1. **Generate** — Create candidates with structural constraints (Notebook 05)
2. **Pre-screen** — Filter by composition validity, bond distances (`script/eval_screen.py`)
3. **MLIP screen** — Evaluate with GNN models or CHGNet
4. **DFT validation** — Final verification of the best candidates

In the published work, this pipeline narrowed 10 million generated structures
down to 24,743 DFT-validated candidates.

---
## Exercise

1. Pick a different structure from the test set. Relax it with CHGNet. How much did the energy change? How much did the atoms move?
2. Try relaxing the **NaCl structure** from Notebook 01 (build it from scratch). Does CHGNet predict the correct lattice parameter?

In [ ]:
# Your code here


---
## Key takeaways

1. **Generation without evaluation is useless.** You need fast, reliable ways to screen candidates.
2. **MLIPs (like CHGNet) are 1000x faster than DFT** and accurate enough for initial screening.
3. **Relaxation** lets you check if a generated structure is near an energy minimum.
4. **SCIGEN's pipeline** combines generation → pre-screening → MLIP evaluation → DFT validation.

In the next notebook, we'll generate crystal structures with SCIGEN!

---
## What's next?

Proceed to **Notebook 05: SCIGEN Generation** — the capstone demo where we generate crystal structures with targeted lattice constraints.